# *Outlines*

<img src="../assets/Flowchart_FatigueAssessment.png" width="700">

# **Introduction**

We will be using loads from [WILLOW Norther Open Source Dataset](https://zenodo.org/records/11093262) for conducting this workshop

<img src="../assets/Fig1_Norther_locaction.png" width="800">



There are various sensors installed on this turbine including accelerometers, strain gauges, fiber bragg gratings (FBGs)


<img src="../assets/Sensor_Channels.png" width="700">


The list of SCADA parameters and the intervals for which the data is avaliable is also given as follows:

<img src="../assets/SCADA_Operational_Data.png" width="500">



# *Loading Necessary Packages*

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import gc

# Packages for handling and processing load data
import pandas as pd
import numpy as np
from tqdm import tqdm
tqdm.pandas() 

# Importing py_fatigue for load signals processing, fatigue damage and crack growth calculations
import py_fatigue as pf


#Packages for Plotting
import plotly.graph_objects as go
import matplotlib.pyplot as plt
import matplotlib.dates as mdates


# Defining SN-Curve

In [ ]:
sn_cp = pf.SNCurve(
    slope=[3., 5.], 
    intercept=[11.764, 15.606], 
    environment = 'Seawater with cathodic protection', 
    curve = 'DNV-D-C', 
    norm ='DNVGL-RP-C203/2016',
    color='g'
)

sn_cp

# *Fatigue Damage Calculation*

In [ ]:
deg230_data = pd.read_pickle("deg230_data.pkl")

In [ ]:
LEF = 3
SEF = 0.8
scale_f = 1.15
SCF_in = 1.1
material_SF = 1.25

damage_Factor = LEF*SEF*SCF_in*scale_f*material_SF

print(damage_Factor)

In [ ]:
cc = deg230_data.iloc[0,0]

In [ ]:
scaled_cc = cc*damage_Factor
scaled_cc

In [ ]:
fat_dmg = pf.damage.stress_life.get_pm(scaled_cc, sn_cp)

In [ ]:
fat_dmg.sum()

In [ ]:
for col in deg230_data.columns:
    if 'CC_' in col:
        damage_col = col + " Damage"
        deg230_data[damage_col] = deg230_data[col].progress_apply(
            lambda cc: (
                sum(pf.damage.stress_life.get_pm(cc * damage_Factor, sn_cp))
                if cc is not None and not isinstance(cc, float) else np.nan
            )
        )


In [ ]:
deg230_data

In [ ]:
deg230_data['CC_TP_SG_LAT014_DEG230_0_nr1 [MPa] Damage']

In [ ]:
deg230_data['cumulative damage'] = deg230_data['CC_TP_SG_LAT014_DEG230_0_nr1 [MPa] Damage'].cumsum()
deg230_data

Plot the cumulative damage with time to see the damage accumulation rates

In [ ]:
plt.figure(figsize=(10,3))
plt.plot()

In [ ]:
# Synthetic timeline
n_steps = len(deg230_data)
new_index = pd.date_range(start="2020-07-03 01:30:00+00:00", periods=n_steps, freq="10min")
deg230_data = deg230_data.set_index(new_index)

In [ ]:
damage_cols = [c for c in deg230_data.columns if "Damage" in c]


deg230_data['cumulative damage'] = deg230_data[damage_cols].cumsum()

# --- Plot ---
fig, ax = plt.subplots(figsize=(10, 4))

for col in damage_cols:
    ax.plot(deg230_data.index, deg230_data['cumulative damage'], label=col, marker='o', linestyle='-')

# Format the x-axis
ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))    # show only hour:minute


plt.setp(ax.get_xticklabels(), rotation=45, ha='right')

ax.set_xlabel("Time")
ax.set_ylabel("Cumulative Fatigue Damage")
ax.set_title("Cumulative Fatigue Damage at 230° Heading")
ax.legend()
ax.grid(True)
plt.tight_layout()
plt.show()


In [ ]:
OS_dmg_mean = deg230_data.groupby('case_ID')['CC_TP_SG_LAT014_DEG230_0_nr1 [MPa] Damage'].mean()
OS_dmg_mean

## **Fatigue Lifetime Estimation**

fatigue life can be estimated as the inverse of Yearly fatige damage

Using OS_dmg_mean['rated']) calculate the yearly expected fatigue damage for the 230° heading in rated condition

Using OS_dmg_mean['parked']) calculate the yearly expected fatigue damage for the 230° heading in parked condition

We can assume proportions of operational conditions for further analysis, we can play with these two operational states to see their effects on the lifetime

In [ ]:
prob_parked = 0.2
prob_rated = 1-prob_parked

Using proportion of each operating condition, we can calculate the overall fatigue damage and fatigue life

In [ ]:
# Your code here to calculate the mean damage and lifetime



# Tasks
1: Change the SN curve from DNV-D-C to DNV-D-Air with slope=[3, 5], intercept=[12.164, 15.606]; change the environment description to Air. 

2: Change the Load Extrapolation Factor LEF to 1 and SCF to 1.5

3: Plot Cumulative Fatigue Damage Accumulation at 230 Degree Heading and check mean damage for Parked and Rated conditions, what do you notice, quantitatively and qualitatively?

4: Assuming the turbine remains 15% Parked with LEF = 1 and SCF = 1.5 and DNV-D-A Curve, what lifetime should we expect at this location.

5: Any suggestions on how could we improve this analysis?

# Crack Growth Calculations

In [ ]:
t = 80.0
d = 6.50*1000
crack_depth = 1.5

In [ ]:
geo = pf.geometry.HollowCylinder(
  initial_depth=crack_depth,
  thickness=t,
  height=100.,
  outer_diameter=d,
  crack_position="external"
)
geo.plot()
geo

In [ ]:
SIF = np.linspace(1,2500, 300)
SLOPE = np.array([2.88, 5.1, 8.16, 5.1, 2.88])
INTER = np.array([1E-16, 1E-20, 1E-27, 1E-19, 1E-13])
THRESHOLD = 20
CRITICAL = 2000
pc = pf.ParisCurve(slope=SLOPE, intercept=INTER, threshold=THRESHOLD,
                   critical=CRITICAL, norm="The norm",
                   environment="Environment", curve="nr.")
pc.plot()


In [ ]:
cg_d = pf.damage.crack_growth.get_crack_growth(
  scaled_cc, pc, geo, express_mode=True
)
print(f"Cycles to end: {int(cg_d.final_cycles)}")

In [ ]:
cg_d.crack_depth.max()

In [ ]:
plt.plot(np.cumsum(cg_d.count_cycle), cg_d.crack_depth)
plt.xlabel("Cumulative cycles")
plt.ylabel("Crack size, mm")
plt.show()

In [ ]:
plt.plot(np.cumsum(cg_d.count_cycle),
         cg_d.crack_depth - cg_d.crack_depth[0])

plt.xlabel("Cumulative cycles")
plt.ylabel("Δ Crack size (mm)")   # shows relative growth
plt.show()